# 🤖 AI-Based Intrusion Detection System
## NSL-KDD Dataset Exploration & Model Training

This notebook walks through:
1. Loading and exploring the NSL-KDD dataset
2. Feature engineering and preprocessing
3. Training Random Forest and Decision Tree classifiers
4. Evaluating models with SOC-relevant metrics
5. Visualizing feature importance

**Run cells in order from top to bottom.**

In [ ]:
# Install dependencies if needed
# !pip install pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, RocCurveDisplay, accuracy_score
)

print('Libraries loaded successfully!')

## 1. Load NSL-KDD Dataset

In [ ]:
COLUMNS = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes',
    'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
    'num_shells','num_access_files','num_outbound_cmds','is_host_login',
    'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
    'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty'
]

# Load dataset (download first — see data/README.md)
try:
    df = pd.read_csv('../data/raw/KDDTrain+.txt', header=None, names=COLUMNS)
    print(f'Loaded {len(df):,} records')
    print(df.head())
except FileNotFoundError:
    print('Dataset not found. Generating synthetic data for demo...')
    # Generate synthetic demo data
    np.random.seed(42)
    n = 2000
    df_demo = pd.DataFrame({
        'src_bytes': np.concatenate([np.random.exponential(500, n//2), np.random.exponential(50000, n//2)]),
        'dst_bytes': np.random.exponential(1000, n),
        'duration': np.concatenate([np.random.exponential(30, n//2), np.random.exponential(1, n//2)]),
        'count': np.concatenate([np.random.poisson(5, n//2), np.random.poisson(200, n//2)]),
        'serror_rate': np.concatenate([np.random.uniform(0, 0.1, n//2), np.random.uniform(0.7, 1.0, n//2)]),
        'label': ['normal']*(n//2) + ['neptune']*(n//2)
    })
    df = df_demo
    print(f'Generated {len(df)} synthetic records for demo')

## 2. Exploratory Data Analysis

In [ ]:
# Attack type distribution
plt.figure(figsize=(14, 5))
label_counts = df['label'].value_counts().head(15)
bars = plt.bar(label_counts.index, label_counts.values, 
               color=['green' if l == 'normal' else 'red' for l in label_counts.index])
plt.title('Traffic Label Distribution (NSL-KDD)', fontsize=14)
plt.xlabel('Label')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('\nTop 15 Labels:')
print(label_counts)

In [ ]:
# Binary label distribution (Normal vs Attack)
df['label_binary'] = df['label'].apply(lambda x: 0 if x.strip() == 'normal' else 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
counts = df['label_binary'].value_counts()
axes[0].pie(counts.values, labels=['Normal', 'Attack'], autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'])
axes[0].set_title('Binary Class Distribution')

# Src bytes comparison
if 'src_bytes' in df.columns:
    df_plot = df[df['src_bytes'] < df['src_bytes'].quantile(0.99)]
    normal_bytes = df_plot[df_plot['label_binary']==0]['src_bytes']
    attack_bytes = df_plot[df_plot['label_binary']==1]['src_bytes']
    axes[1].hist(normal_bytes, bins=50, alpha=0.6, label='Normal', color='green')
    axes[1].hist(attack_bytes, bins=50, alpha=0.6, label='Attack', color='red')
    axes[1].set_title('src_bytes Distribution: Normal vs Attack')
    axes[1].set_xlabel('src_bytes')
    axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
# Encode categorical features
cat_cols = ['protocol_type', 'service', 'flag']
df_processed = df.copy()

for col in cat_cols:
    if col in df_processed.columns:
        le = LabelEncoder()
        df_processed[col] = le.fit_transform(df_processed[col].astype(str))
        print(f'Encoded {col}: {len(le.classes_)} unique values')

# Select features
drop_cols = ['label', 'label_binary', 'difficulty']
feature_cols = [c for c in df_processed.columns if c not in drop_cols]

X = df_processed[feature_cols].fillna(0).values
y = df_processed['label_binary'].values

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'\nFeature matrix shape: {X_scaled.shape}')
print(f'Labels: {y.shape}')

## 4. Train Models

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print('Random Forest trained!')

# Train Decision Tree
dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X_train, y_train)
print('Decision Tree trained!')

## 5. Evaluate Models

In [ ]:
for name, model in [('Random Forest', rf), ('Decision Tree', dt)]:
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f'\n=== {name} ===')
    print(f'Accuracy: {acc:.4f} ({acc*100:.2f}%)')
    print(classification_report(y_test, y_pred, target_names=['Normal', 'Attack']))

In [ ]:
# Confusion Matrix Heatmaps
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, model) in zip(axes, [('Random Forest', rf), ('Decision Tree', dt)]):
    cm = confusion_matrix(y_test, model.predict(X_test))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Attack'],
                yticklabels=['Normal', 'Attack'], ax=ax)
    ax.set_title(f'{name} — Confusion Matrix')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance (Random Forest)
importances = rf.feature_importances_
top_n = 15
sorted_idx = np.argsort(importances)[::-1][:top_n]
top_features = [feature_cols[i] for i in sorted_idx]
top_importances = importances[sorted_idx]

plt.figure(figsize=(10, 6))
plt.barh(top_features[::-1], top_importances[::-1], color='steelblue')
plt.xlabel('Feature Importance Score')
plt.title('Top 15 Features — Random Forest IDS')
plt.tight_layout()
plt.show()

print('\nTop 15 Most Important Features:')
for i, (feat, imp) in enumerate(zip(top_features, top_importances), 1):
    print(f'  {i:2}. {feat:<35} {imp:.4f}')

## 6. Real-Time Prediction Simulation

In [ ]:
# Simulate predicting a single connection
sample_idx = 42
sample = X_test[sample_idx].reshape(1, -1)
actual = y_test[sample_idx]

pred = rf.predict(sample)[0]
prob = rf.predict_proba(sample)[0]

print('=== Single Connection Prediction ===')
print(f'Actual Label     : {"Normal" if actual == 0 else "ATTACK"}')
print(f'Predicted Label  : {"Normal" if pred == 0 else "ATTACK"}')
print(f'Confidence       : Normal={prob[0]:.2%}  Attack={prob[1]:.2%}')
print(f'Alert Generated  : {"YES ⚠️" if pred == 1 else "No"}')